# HLS × ECOSTRESS fused time series — literature ground-truth sites

Runs the cover-crop detection pipeline on **every ground-truth site in `SITES`, one after another**:
a set of commercial fields with known cover-crop history from a published
study, stored in `cc_groundtruth.gpkg` (`fields`, `sites`, `cc_seasons`).

Every cell loops over the sites. For each site the ROI, period, window
length and whether ECOSTRESS is attempted are derived from that site's
labels, never typed in; each site keeps its own working directory and its
own archive folder on the Data Store, and Cell 16 puts all of them on one
page.

### Design rules (unchanged from v5)

**Every cell persists its result to disk and the next cell loads it from
disk.** Restart-safe: any cell runs on its own.

**The container's disk is temporary; the Data Store is not.** Every downloaded
scene is mirrored to the Data Store the moment it is written, so a container
that dies half-way through a 90-minute download loses nothing.

**Labels are read, never inferred here.** HLS NDVI is used to evaluate the
labels' separability, not to create or adjust labels.

### Products written (per site)

| File | Contents |
|---|---|
| `roi.geojson`, `windows.csv` | ROI (fields + 1 km) and the window definitions |
| `fields_site.gpkg`, `seasons_site.csv` | this site's fields and field × season labels |
| `grid_30m.nc`, `grid_70m.nc` | reference grids spanning the ROI |
| `cache/hls_v2/*.nc`, `cache/eco/*.nc` | per-scene caches, mirrored to the Data Store |
| `hls_scenes.nc`, `fused_cube.nc` | per-scene stack and the fused cube |
| `field_id.tif`, `cropland.tif` | ground-truth fields burned onto the 30 m grid |
| `features_by_season.csv` | one row per field × season: features + label |
| `figures/*.png` | QC maps, trajectories, evaluation |


In [ ]:
# =============================================================================
# Cell 0 — Install what a fresh container is missing (idempotent)
# =============================================================================
import importlib, subprocess, sys

NEEDED = ["geopandas", "rioxarray", "earthaccess", "scipy", "h5py", "netCDF4",
          "pyogrio"]
missing = [m for m in NEEDED if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", ", ".join(missing))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--user",
                        *missing], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-2000:]
    importlib.invalidate_caches()
still = [m for m in NEEDED if importlib.util.find_spec(m) is None]
assert not still, f"still missing after install: {still}"
print("environment OK:", ", ".join(NEEDED))

## Cell 1 — Configuration

`SITES` lists the ground-truth sites to process, in order. Every later cell
loops over that list: each site gets its own ROI (its fields + 1 km — so the
HLS download is only the tiles around those fields), its own period, its own
window length and its own working/archive folders. Window length is decided by
data availability, window by window: 16 days while Landsat 8 alone feeds HLS,
8 days from the day S30 comes online (late 2015).

In [ ]:
# =============================================================================
# Cell 1 — Imports, configuration, helpers
# =============================================================================
import os, re, json, time, gc, shutil, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
from shapely.geometry import box, Polygon
from rasterio.enums import Resampling
from rasterio import features
import rasterio
import earthaccess
import warnings; warnings.filterwarnings("ignore")

SITES = ["IN_PENG2025", "IL_WELCH2016_CG", "NE_BLANCO2017"]   # processed in this order

CFG = dict(
    # --- ground truth ----------------------------------------------------
    gt_gpkg   = "/data-store/iplant/home/pengyuwh/CoverCrop_Fusion/ground_truth/"
                "cc_groundtruth.gpkg",
    roi_buffer_m = 1000,        # each site's fields' bounding box grown by this much

    # --- time ------------------------------------------------------------
    # The period is derived from the labels: first detect_start minus lead,
    # last detect_end plus tail (the tail keeps the cash-crop emergence that
    # term_drop needs as its "after" state).
    lead_days = 45,
    tail_days = 75,
    x_days_4sat = 8,            # window once L30 + S30 are both flying
    x_days_1sat = 16,           # window while only Landsat 8 feeds HLS
    s30_start   = "2015-12-01", # HLS S30 v2.0 begins here (approx.)
    eco_start   = "2018-07-09", # first ECOSTRESS acquisitions

    # --- resolution / products ------------------------------------------
    res_optical = 30,
    res_thermal = 70,
    eco_product = "ECO_L3T_JET",
    eco_var     = "ET",
    cloud_max   = 80.0,
    utc_offset  = -6,           # overwritten per site from the longitude

    # --- seasons ---------------------------------------------------------
    # None = every usable season of the site. NE_BLANCO2017 carries eight
    # extra single-field seasons (2017-25, all "no CC") that would stretch the
    # run to twelve years for a few negatives; the trial years are enough.
    seasons = {"NE_BLANCO2017": ["2013-14", "2014-15", "2015-16"]},

    # --- storage ---------------------------------------------------------
    work_root    = "/home/jovyan/fusion_sites",                      # local disk
    archive_root = "/data-store/iplant/home/pengyuwh/CoverCrop_Fusion/ground_truth_runs",
    use_archive  = True,
)

# Per-site globals. Every cell below starts its loop with use_site(SITE), which
# points WORK / ARCHIVE / caches / figures at that site and loads the site's own
# period and window settings (cfg_site.json, written by Cell 3) into CFG.
SITE = WORK = ARCHIVE = CACHE_HLS = CACHE_ECO = FIG_DIR = None
P = lambda name: os.path.join(WORK, name)


def use_site(site, quiet=False):
    global SITE, WORK, ARCHIVE, CACHE_HLS, CACHE_ECO, FIG_DIR
    SITE      = site
    WORK      = os.path.join(CFG["work_root"], site)
    ARCHIVE   = os.path.join(CFG["archive_root"], site)
    CACHE_HLS = os.path.join(WORK, "cache", "hls_v2")
    CACHE_ECO = os.path.join(WORK, "cache", "eco")
    FIG_DIR   = os.path.join(WORK, "figures")
    for d in (WORK, CACHE_HLS, CACHE_ECO, FIG_DIR):
        os.makedirs(d, exist_ok=True)
    if os.path.isfile(P("cfg_site.json")):
        CFG.update(json.load(open(P("cfg_site.json"))))
    if not quiet:
        print(f"\n{'─'*12} {site} {'─'*12}")


def have(name):
    """True when a product exists in the current site's working dir."""
    p = P(name)
    return os.path.isfile(p) and os.path.getsize(p) > 0


def load_da(path, name=None):
    """Open a NetCDF written by this notebook and return it as a DataArray.

    Reads fully into memory and CLOSES the file (a lazily-open NetCDF keeps an
    HDF5 lock that turns a later rewrite into a bare PermissionError), selects
    the variable by name (write_crs() adds a spatial_ref variable, so a guessing
    reader would refuse), and restores the CRS that to_netcdf demoted to an
    attribute.
    """
    wkt = None
    with xr.open_dataset(path) as ds:
        if name is None:
            name = next(v for v in ds.data_vars if v != "spatial_ref")
        da = ds[name].load()
        if da.rio.crs is None and "spatial_ref" in ds:
            att = ds["spatial_ref"].attrs
            wkt = att.get("crs_wkt") or att.get("spatial_ref")
    return da.rio.write_crs(wkt) if wkt else da


def mirror(local_path):
    """Copy one freshly written cache file to the current site's Data Store
    folder (sequential copy — the one write pattern the FUSE mount handles
    well). Returns the mirror path."""
    rel = os.path.relpath(local_path, WORK)
    dst = os.path.join(ARCHIVE, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(local_path, dst)
    return dst


# --- restore every site from its archive -----------------------------------
for SITE in SITES:
    use_site(SITE, quiet=True)
    RESTORED = []
    if CFG["use_archive"] and os.path.isdir(ARCHIVE):
        for root, _, files in os.walk(ARCHIVE):
            rel = os.path.relpath(root, ARCHIVE)
            for f in sorted(files):
                dst = P(f) if rel == "." else P(os.path.join(rel, f))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, f), dst)
                    RESTORED.append(os.path.relpath(dst, WORK))
    top = [f for f in RESTORED if os.sep not in f]
    print(f"{SITE:<18} restored from archive: {len(top)} product(s), "
          f"{len(RESTORED) - len(top)} cached scene(s)")

# Prove the working disk really accepts a NetCDF write.
_probe = os.path.join(CFG["work_root"], ".write_test.nc")
if os.path.exists(_probe):
    os.remove(_probe)
xr.DataArray(np.zeros((2, 2), "float32"), dims=("y", "x")).to_netcdf(_probe)
os.remove(_probe)
print(f"\nsites       : {', '.join(SITES)}")
print(f"working dir : {CFG['work_root']}/<site>")
print(f"archive     : {CFG['archive_root']}/<site>")
print("NetCDF write test: OK")

## Cell 2 — Earthdata authentication

`persist=True` writes `~/.netrc` on first use. When the archive already holds
every download product and no login is available, the cell records
`auth = None` and the download cells skip.

In [ ]:
# =============================================================================
# Cell 2 — Authenticate
# =============================================================================
try:
    auth = earthaccess.login(persist=True)
    assert auth.authenticated
    # GDAL reads the HLS GeoTIFFs directly over HTTPS with this bearer token
    # (Cell 6); no per-file Python handles, and thread-safe.
    GDAL_OPTS = dict(GDAL_HTTP_HEADERS=f"Authorization: Bearer {auth.token['access_token']}",
                     GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
                     CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
                     GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="3")
    print("Earthdata: authenticated (bearer token handed to GDAL)")
except Exception as e:
    if have("hls_scenes.nc") and have("fused_cube.nc"):
        auth, GDAL_OPTS = None, {}
        print(f"Earthdata login unavailable ({type(e).__name__}) — continuing "
              f"with archived products; nothing will be downloaded")
    else:
        raise RuntimeError("Earthdata login failed and no archived products "
                           "to fall back on — create ~/.netrc or log in") from e

## Cell 3 — Site, labels, ROI, period, windows

Everything here comes out of the ground-truth store. Only labels with
`usable_for_rs` and a known `cc_present` (0/1) are kept; −1 rows are unknown
and never enter an evaluation.

Windows are half-open `[start, next)` and tile the period with no gap. The
window length is 8 days when the period starts after S30 came online, 16 days
otherwise.

In [ ]:
# =============================================================================
# Cell 3 — Load the site's fields and labels; derive ROI, period, windows
# =============================================================================
def make_windows(start, end, split, x_before, x_after):
    """Consecutive HALF-OPEN windows tiling [start, end]: `x_before` days long
    before `split` (Landsat only), `x_after` days from `split` on (L30 + S30).
    A window never straddles `split`."""
    start, end, split = pd.Timestamp(start), pd.Timestamp(end), pd.Timestamp(split)
    end_excl = end + pd.Timedelta(days=1)
    edges = []
    for seg_start, seg_end, x in ((start, min(split, end_excl), x_before),
                                  (max(split, start), end_excl, x_after)):
        if seg_start >= seg_end:
            continue
        e = pd.date_range(seg_start, seg_end - pd.Timedelta(days=1), freq=f"{x}D")
        edges += [(ei, min(ei + pd.Timedelta(days=x), seg_end)) for ei in e]
    w = pd.DataFrame([dict(win_id=i, win_start=s, win_next=n, win_end=n - pd.Timedelta(days=1),
                           win_center=s + (n - s) / 2, win_days=(n - s).days)
                      for i, (s, n) in enumerate(edges)])
    assert (w.win_next.values[:-1] == w.win_start.values[1:]).all()   # no gaps, no overlap
    assert w.win_start.iloc[0] == start and w.win_next.iloc[-1] == end_excl
    return w


for SITE in SITES:
    use_site(SITE)
    fields_all  = gpd.read_file(CFG["gt_gpkg"], layer="fields")
    seasons_all = pd.DataFrame(gpd.read_file(CFG["gt_gpkg"], layer="cc_seasons"))
    sites_all   = gpd.read_file(CFG["gt_gpkg"], layer="sites")

    fields = fields_all[fields_all.site_id == SITE].reset_index(drop=True)
    assert len(fields), f"no fields for site {SITE} — check CFG['site']"
    seasons = seasons_all[seasons_all.site_id == SITE].copy()
    seasons["usable"] = seasons.usable_for_rs.astype(str).str.lower().isin(["true", "1"])
    labels = seasons[seasons.usable & (seasons.cc_present >= 0)].copy()
    keep_seasons = (CFG["seasons"] or {}).get(SITE)
    if keep_seasons:
        labels = labels[labels.season.isin(keep_seasons)]
    for c in ("detect_start", "detect_end", "best_start", "best_end",
              "cc_seed_date", "cc_termination_date", "next_cash_plant"):
        labels[c] = pd.to_datetime(labels[c], errors="coerce")
    assert len(labels), f"no usable labels for {SITE}"

    # --- ROI: fields' bounding box grown by the buffer, in a metric CRS ---------
    utm = fields.estimate_utm_crs()
    b = fields.to_crs(utm).total_bounds
    roi = gpd.GeoDataFrame({"name": [SITE]},
                           geometry=[box(b[0] - CFG["roi_buffer_m"], b[1] - CFG["roi_buffer_m"],
                                         b[2] + CFG["roi_buffer_m"], b[3] + CFG["roi_buffer_m"])],
                           crs=utm).to_crs("EPSG:4326")
    roi.to_file(P("roi.geojson"), driver="GeoJSON")
    fields.to_file(P("fields_site.gpkg"), driver="GPKG")
    labels.to_csv(P("seasons_site.csv"), index=False)
    CFG["utc_offset"] = int(round(roi.geometry.centroid.x.iloc[0] / 15))

    # --- period and window length from the labels ------------------------------
    date_start = labels.detect_start.min() - pd.Timedelta(days=CFG["lead_days"])
    date_end   = labels.detect_end.max()   + pd.Timedelta(days=CFG["tail_days"])
    CFG.update(date_start=str(date_start.date()), date_end=str(date_end.date()),
               eco_possible=bool(date_end >= pd.Timestamp(CFG["eco_start"])))




    windows = make_windows(CFG["date_start"], CFG["date_end"], CFG["s30_start"],
                           CFG["x_days_1sat"], CFG["x_days_4sat"])
    x_days = int(windows.win_days.mode().iloc[0])            # dominant length, for labels
    CFG["x_days"] = x_days
    windows.to_csv(P("windows.csv"), index=False)
    json.dump({k: v for k, v in CFG.items()}, open(P("cfg_site.json"), "w"), indent=2, default=str)

    w_km = (b[2] - b[0] + 2 * CFG["roi_buffer_m"]) / 1000
    h_km = (b[3] - b[1] + 2 * CFG["roi_buffer_m"]) / 1000
    print(f"site     : {SITE}   fields {len(fields)}  "
          f"({', '.join(sorted(fields.role.unique()))})")
    print(f"labels   : {len(labels)} usable field-seasons  "
          f"({int((labels.cc_present == 1).sum())} CC / {int((labels.cc_present == 0).sum())} no CC)"
          f"  seasons {sorted(labels.season.unique())}")
    print(f"ROI      : {w_km:.1f} x {h_km:.1f} km   ({utm.name})")
    print(f"period   : {CFG['date_start']} → {CFG['date_end']}   {len(windows)} windows "
          + " + ".join(f"{n} x {d} d" for d, n in windows.win_days.value_counts().sort_index(ascending=False).items()))
    print(f"ECOSTRESS: {'possible' if CFG['eco_possible'] else 'not available for this period'}")
    print(f"saved    : roi.geojson, fields_site.gpkg, seasons_site.csv, windows.csv, cfg_site.json")

## Cell 4 — Reference grids

Built from the ROI polygon, never from whichever scene arrives first (a
scene-derived grid inherits that scene's footprint and silently truncates
everything else). Both grids share one CRS and origin.

In [ ]:
# =============================================================================
# Cell 4 — Build the 30 m and 70 m reference grids from the ROI
# =============================================================================
def make_grid(roi, res, crs):
    """An empty raster spanning the ROI at `res` metres, derived from the ROI."""
    x0, y0, x1, y1 = roi.to_crs(crs).total_bounds
    nx, ny = int(np.ceil((x1 - x0) / res)), int(np.ceil((y1 - y0) / res))
    return xr.DataArray(
        np.full((ny, nx), np.nan, "float32"),
        coords={"y": y1 - (np.arange(ny) + 0.5) * res,
                "x": x0 + (np.arange(nx) + 0.5) * res},
        dims=("y", "x"), name="grid").rio.write_crs(crs)


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson"))
    utm = roi.estimate_utm_crs()




    grid30 = make_grid(roi, CFG["res_optical"], utm)
    grid70 = make_grid(roi, CFG["res_thermal"], utm)
    # netCDF4 refuses to overwrite a file another process holds open; unlink first.
    for path in (P("grid_30m.nc"), P("grid_70m.nc")):
        if os.path.exists(path):
            os.remove(path)
    grid30.to_netcdf(P("grid_30m.nc"))
    grid70.to_netcdf(P("grid_70m.nc"))

    rx0, ry0, rx1, ry1 = roi.to_crs(utm).total_bounds
    for g, res in ((grid30, CFG["res_optical"]), (grid70, CFG["res_thermal"])):
        assert g.sizes["x"] * res >= (rx1 - rx0) and g.sizes["y"] * res >= (ry1 - ry0)
        print(f"{res:>3} m grid: {g.sizes['y']:>4} x {g.sizes['x']:<4} px")
    print(f"CRS: {utm.name}\nsaved: grid_30m.nc, grid_70m.nc")

## Cell 5 — Find HLS scenes

`HLSL30` (Landsat 8/9) and `HLSS30` (Sentinel-2 A/B) are already harmonised by
NASA — one 30 m grid, one atmospheric correction, BRDF-normalised, with
Sentinel-2 bands adjusted to Landsat band-passes. Combining them into a single
series is the entire purpose of the product.

Two details that fail without an error message:

- **`HLSF30` does not exist.** CMR ignores an unknown short name, halving your
  temporal density while appearing to succeed.
- **NIR differs by sensor**: Landsat `B05`, Sentinel-2 **`B8A`** (narrow NIR,
  865 nm) — *not* `B08`. NASA harmonised against B8A; B08 introduces a bias.

Scene cloud cover lives in `AdditionalAttributes.CLOUD_COVERAGE`, not in the
standard UMM `CloudCover` field. Missing metadata is treated as 0% (keep the
scene) — per-pixel Fmask does the real masking, so an unreadable field must not
throw away a usable observation.

In [ ]:
# =============================================================================
# Cell 5 — Search HLS, resolve band URLs, save the index
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    CFG.update(json.load(open(P("cfg_site.json"))))     # period from Cell 3
    roi = gpd.read_file(P("roi.geojson"))
    bbox = tuple(roi.total_bounds)

    # NIR and SWIR band ids differ by sensor. Getting one wrong raises nothing --
    # it just yields an index with no physical meaning.
    HLS_BANDS = {"L30": {"red": "B04", "nir": "B05",     # Landsat 8/9
                         "swir1": "B06", "swir2": "B07"},
                 "S30": {"red": "B04", "nir": "B8A",     # Sentinel-2: B8A, not B08
                         "swir1": "B11", "swir2": "B12"}}

    if globals().get("auth") is None and have("hls_index.csv"):
        hls_index = pd.read_csv(P("hls_index.csv"), parse_dates=["date"])
        print(f"Earthdata offline — using archived hls_index.csv "
              f"({len(hls_index)} scenes, {hls_index.date.min().date()} → "
              f"{hls_index.date.max().date()})")
        granules = None
    else:
        granules = earthaccess.search_data(
            short_name=["HLSL30", "HLSS30"],   # HLSF30 is not a real product
            bounding_box=bbox,
            temporal=(CFG["date_start"], CFG["date_end"]),
            count=-1)                          # -1 = all; a fixed count truncates
        assert granules, "no HLS granules found — check bbox, dates, login"

        def granule_cover(g, roi_geom):
            """Fraction of the ROI inside the granule's footprint (0-1)."""
            try:
                geom = g["umm"]["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]
                if "GPolygons" in geom:
                    pts = geom["GPolygons"][0]["Boundary"]["Points"]
                    poly = Polygon([(p["Longitude"], p["Latitude"]) for p in pts])
                else:
                    bb = geom["BoundingRectangles"][0]
                    poly = box(bb["WestBoundingCoordinate"], bb["SouthBoundingCoordinate"],
                               bb["EastBoundingCoordinate"], bb["NorthBoundingCoordinate"])
                return poly.intersection(roi_geom).area / roi_geom.area
            except Exception:
                return np.nan

        roi_geom = roi.geometry.iloc[0]
        rows = []
        for g in granules:
            parts = g["meta"]["native-id"].split(".")     # HLS.S30.T16TDK.2024274T16...
            sensor, tile = parts[1], parts[2]
            attrs = {a.get("Name", ""): a.get("Values", [None])[0]
                     for a in g["umm"].get("AdditionalAttributes", [])}
            try:
                cloud = float(attrs.get("CLOUD_COVERAGE"))
            except (TypeError, ValueError):
                cloud = np.nan                            # unknown ≠ cloudy
            if np.nan_to_num(cloud, nan=0.0) > CFG["cloud_max"]:
                continue

            links = g.data_links()
            pick = lambda s: next((l for l in links if l.endswith(f".{s}.tif")), None)
            bands = {k: pick(v) for k, v in HLS_BANDS[sensor].items()}
            fmask = pick("Fmask")
            if not all(bands.values()) or not fmask:
                continue
            rows.append(dict(gid=g["meta"]["native-id"], sensor=sensor, tile=tile,
                             date=pd.to_datetime(parts[3][:7], format="%Y%j"),
                             cloud=cloud, cover=granule_cover(g, roi_geom),
                             fmask=fmask, **bands))

        hls_index = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
        assert len(hls_index), "no granule had a complete red/NIR/SWIR/Fmask set"

        # A small ROI usually sits inside one MGRS tile; where it sits in the
        # overlap of two, both tiles carry the same pixels from the same overpass,
        # so one of them is enough. Keep the fully-covering tile with the most
        # granules; keep every tile only when none covers the ROI on its own.
        cov = hls_index.groupby("tile").cover.median()
        full = cov[cov >= 0.99].index
        n_all = len(hls_index)
        if len(full):
            keep = hls_index[hls_index.tile.isin(full)].tile.value_counts().idxmax()
            hls_index = hls_index[hls_index.tile == keep].reset_index(drop=True)
        hls_index.to_csv(P("hls_index.csv"), index=False)

        print(f"granules found : {len(granules)}  →  complete band sets {n_all}  "
              f"→  after tile selection {len(hls_index)}")
        print(f"ROI cover/tile : {', '.join(f'{k} {v:.0%}' for k, v in cov.items())}")
        print(f"sensors        : {dict(hls_index.sensor.value_counts())}")
        print(f"MGRS tiles     : {dict(hls_index.tile.value_counts())}")
        print(f"dates          : {hls_index.date.min().date()} → {hls_index.date.max().date()}")
        print(f"saved          : hls_index.csv")

## Cell 6 — Download HLS and compute NDVI

**Fmask bits (HLS v2.0):** 0 cirrus · 1 cloud · 2 adjacent to cloud · 3 cloud
shadow · 4 snow/ice · 5 water · 6–7 aerosol level.

Bits 0–5 are masked; bits 6–7 are deliberately left alone — masking the aerosol
level discards a great many good observations. Snow (bit 4) matters here:
unmasked winter snow reads as low NDVI and imitates bare soil.

Each scene is saved as NetCDF **with its coordinates**, so the grid can be
changed later without re-downloading. NDVI is a ratio, so reflectance scale
factors cancel — no scaling needed.

Each scene is mirrored to the Data Store the moment it is written, so a run
interrupted by a container restart resumes from the mirror, not from zero.

**Why it takes time at all.** Each scene clipped to a 3 km ROI is ~100 × 100
pixels — about 100 KB across five bands. The cost is not bytes but round
trips: every band is a separate HTTPS request to a cloud-optimised GeoTIFF,
and the wait is latency. Two things cut it: Fmask is read **first**, and the
four reflectance bands are only requested when at least `MIN_CLEAR` of the ROI
is clear (a clouded-out scene costs one request and leaves a `.skip` marker);
and `N_WORKERS` scenes are fetched concurrently, since threads waiting on the
network cost nothing.

*Safe to interrupt; re-running skips whatever is already cached or marked.*

In [ ]:
# =============================================================================
# Cell 6 — Download each scene, compute NDVI, cache as NetCDF (resumable)
# =============================================================================
def open_clip(url, roi, masked=True):
    """Open a (remote) cloud-optimised GeoTIFF by URL and read only the ROI
    window, in the file's own CRS. GDAL does the HTTPS range requests itself
    (bearer token from Cell 2), so no Python file object is involved and the
    call is safe to run from several threads at once."""
    with rxr.open_rasterio(url, masked=masked) as da:
        da = da.squeeze(drop=True)
        return da.rio.clip(roi.to_crs(da.rio.crs).geometry, from_disk=True).load()

import threading

def norm_diff(x, y):
    """(x-y)/(x+y), clipped to the physically possible range."""
    v = (x - y) / (x + y)
    return v.where((v >= -1) & (v <= 1))

def fetch_scene(r, cache_dir, grid, roi, max_retry=3):
    """One scene: Fmask first; bands only if enough of the ROI is clear.

    Returns (status, error): status is "hit" (already cached), "skipped"
    (clouded out — a .skip marker is written so it is never fetched again),
    "new" or "failed".
    """
    path = os.path.join(cache_dir, r.gid + ".nc")        # granule id = unique
    skip = path[:-3] + ".skip"
    if os.path.exists(path) or os.path.exists(skip):
        return "hit", None
    err = None
    for attempt in range(max_retry):
        try:
            with rasterio.Env(**GDAL_OPTS):
                fmask = open_clip(r.fmask, roi, masked=False).rio.reproject_match(grid)
                clear = (np.nan_to_num(fmask.values, nan=255).astype("uint8") & FMASK_BAD) == 0
                if clear.mean() < MIN_CLEAR:
                    with open(skip, "w") as f:
                        f.write(f"clear fraction {clear.mean():.3f}\n")
                    mirror(skip)
                    return "skipped", None

                red, nir = open_clip(r.red, roi), open_clip(r.nir, roi)
                sw1, sw2 = open_clip(r.swir1, roi), open_clip(r.swir2, roi)
                # Always reproject onto the reference grid. Comparing shapes alone
                # is unsafe: two MGRS tiles can clip to identical shapes with
                # different origins, and that misalignment is invisible once stored.
                ndvi = norm_diff(nir, red).rio.reproject_match(grid)   # green vegetation
                ndti = norm_diff(sw1, sw2).rio.reproject_match(grid)   # residue vs bare soil
                clear = xr.DataArray(clear, coords=ndvi.coords, dims=ndvi.dims)
                ds = xr.Dataset({"ndvi": ndvi.where(clear), "ndti": ndti.where(clear)})
                # Clear the attributes ON EACH VARIABLE, not just on the Dataset.
                # HLS reflectance carries scale_factor=0.0001; the indices are
                # unitless ratios, but the attribute rides along per variable and
                # xarray re-applies it on the next read, shrinking every value by
                # 1e4 with no warning.
                for v in ("ndvi", "ndti"):
                    ds[v].attrs = {}
                    ds[v].encoding = {}
                ds.attrs = dict(gid=r.gid, sensor=r.sensor, date=str(r.date.date()))
                with NC_LOCK:               # one NetCDF write at a time
                    ds.to_netcdf(path)
                mirror(path)                # Data Store copy: survives the container
                return "new", None
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(5 * 2 ** attempt)                   # exponential back-off
    return "failed", err

def download_hls(index, cache_dir, grid, roi):
    """Fetch every scene of `index` with N_WORKERS threads (resumable)."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t0 = time.time()
    counts = dict(new=0, hit=0, skipped=0, failed=0)
    failed = []
    rows = [r for _, r in index.iterrows()]
    with ThreadPoolExecutor(N_WORKERS) as ex:
        futs = {ex.submit(fetch_scene, r, cache_dir, grid, roi): r.gid for r in rows}
        for k, f in enumerate(as_completed(futs), 1):
            status, err = f.result()
            counts[status] += 1
            if err:
                failed.append(dict(gid=futs[f], error=err[:80]))
            fetched = counts["new"] + counts["skipped"] + counts["failed"]
            if fetched in (12, 50) or (fetched and fetched % 200 == 0):
                rate = fetched / (time.time() - t0)
                print(f"  {k}/{len(rows)}  new {counts['new']}  cloud-skipped "
                      f"{counts['skipped']}  ({(time.time()-t0)/60:.1f} min, "
                      f"~{(len(rows)-k)/rate/60:.0f} min left)")
    print(f"\nHLS: {counts['new']} new, {counts['skipped']} cloud-skipped, "
          f"{counts['hit']} cached, {counts['failed']} failed "
          f"({(time.time()-t0)/60:.1f} min)")
    if failed:
        print(pd.DataFrame(failed).to_string(index=False))
        print("Re-run this cell to retry only the failures.")
    return failed


for SITE in SITES:
    use_site(SITE)
    hls_index = pd.read_csv(P("hls_index.csv"), parse_dates=["date"])
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(
        gpd.read_file(P("roi.geojson")).estimate_utm_crs())
    roi = gpd.read_file(P("roi.geojson"))

    # bits 0-5 = cirrus, cloud, adjacent, shadow, snow, water. 6-7 = aerosol: keep.
    FMASK_BAD = sum(1 << b for b in range(6))         # 0b00111111 = 63
    assert (64 & FMASK_BAD) == 0 and (128 & FMASK_BAD) == 0, "aerosol bits masked!"




    GDAL_OPTS = globals().get("GDAL_OPTS", {})     # set by Cell 2 after login
    NC_LOCK = threading.Lock()                     # the netCDF/HDF5 library is not thread-safe
    MIN_CLEAR = 0.05   # ROI clear fraction below which the scene is not worth its 4 bands
    N_WORKERS = 6      # scenes fetched at once; the wait is latency, not bandwidth








    if not os.listdir(CACHE_HLS) and have("hls_scenes.nc"):
        # The per-scene cache did not survive the container, but its product did.
        print("per-scene cache is empty and hls_scenes.nc was restored from the "
              "archive — nothing to download.\n"
              "Set CFG['use_archive'] = False to rebuild from raw scenes.")
        hls_failed = []
    else:
        hls_failed = download_hls(hls_index, CACHE_HLS, grid30, roi)
        n_nc = len([f for f in os.listdir(CACHE_HLS) if f.endswith(".nc")])
        n_sk = len([f for f in os.listdir(CACHE_HLS) if f.endswith(".skip")])
        print(f"cache now holds {n_nc} scenes + {n_sk} cloud-skip markers "
              f"of {len(hls_index)} indexed")

## Cell 7 — Stack the scenes into NDVI and NDTI cubes

Loads every cached scene, drops any that are entirely cloud-masked, merges
same-day observations (Landsat and Sentinel-2 sometimes pass on the same day),
and writes `hls_scenes.nc` holding both indices on one time axis.

Read the printed ranges. NDVI should span roughly -0.2 to 1.0; NDTI is a much
narrower index, typically -0.1 to 0.3. An NDVI maximum near 1e-4 means an
inherited `scale_factor` was re-applied on read.

In [ ]:
# =============================================================================
# Cell 7 — Assemble the per-scene NDVI + NDTI stacks
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    files = sorted(f for f in os.listdir(CACHE_HLS) if f.endswith(".nc"))
    VARS = ["ndvi", "ndti"]

    if not files and have("hls_scenes.nc"):
        with xr.open_dataset(P("hls_scenes.nc")) as _s:
            scenes = _s.load()
        print(f"no per-scene cache — using restored hls_scenes.nc")
        print(f"observation days : {scenes.sizes['time']}")
        print(f"grid             : {scenes.sizes['y']} x {scenes.sizes['x']} px")
        for v in VARS:
            arr = scenes[v].values
            print(f"  {v.upper():<5} valid {np.isfinite(arr).mean():.1%}  "
                  f"range {np.nanmin(arr):+.2f} … {np.nanmax(arr):+.2f}")
        files = None
    assert files is None or files, "HLS cache is empty — run Cell 6 first"

    if files:
        stacks, dates, sensors, fracs = {v: [] for v in VARS}, [], [], []
        for fn in files:
            # open_dataset, not a DataArray reader: each file now holds two variables,
            # and a reader that guesses could silently hand back NDTI as NDVI.
            with xr.open_dataset(os.path.join(CACHE_HLS, fn)) as ds:
                frac = float(np.isfinite(ds["ndvi"].values).mean())
                if frac == 0:                       # scene entirely clouded out
                    continue
                for v in VARS:
                    stacks[v].append(ds[v].values.astype("float32"))
            dates.append(pd.to_datetime(fn.split(".")[3][:7], format="%Y%j"))
            sensors.append(fn.split(".")[1])
            fracs.append(frac)

        grid30 = load_da(P("grid_30m.nc"))
        scenes = xr.Dataset(
            {v: (("time", "y", "x"), np.stack(stacks[v])) for v in VARS},
            coords={"time": pd.DatetimeIndex(dates),
                    "y": grid30.y.values, "x": grid30.x.values})

        # Same-day L30 + S30 overpasses are averaged so neither is counted twice
        n_before = scenes.sizes["time"]
        scenes = scenes.groupby("time").mean("time").sortby("time")
        if os.path.exists(P("hls_scenes.nc")):
            os.remove(P("hls_scenes.nc"))           # HDF5 locks an open file
        scenes.to_netcdf(P("hls_scenes.nc"))

        fr = np.array(fracs)
        print(f"scenes loaded    : {len(files)}  →  usable {len(fracs)}")
        print(f"same-day merged  : {n_before - scenes.sizes['time']}")
        print(f"observation days : {scenes.sizes['time']}")
        print(f"grid             : {scenes.sizes['y']} x {scenes.sizes['x']} px")
        for v in VARS:
            arr = scenes[v].values
            print(f"  {v.upper():<5} valid {np.isfinite(arr).mean():.1%}  "
                  f"range {np.nanmin(arr):+.2f} … {np.nanmax(arr):+.2f}")
        print(f"\nper-scene valid fraction (after cloud masking):")
        print(f"  median {np.median(fr):.1%} | >90%: {(fr>0.9).sum()} | "
              f"50-90%: {((fr>0.5)&(fr<=0.9)).sum()} | <50%: {(fr<=0.5).sum()}")
        print(f"  → low fractions are cloud/shadow/snow; scenes below MIN_CLEAR never "
              f"reached this cell")
        print(f"\nsaved: hls_scenes.nc")

## Cell 8 — Find ECOSTRESS scenes (only when the period allows)

ECOSTRESS began acquiring in July 2018. For a site whose labels end before
that (NE_BLANCO2017: 2013–16) there is nothing to search; the cell records the
fact and the fusion cell builds an optical-only cube.

`ECO_L3T_JET` is daily-integrated ET, so the ISS's drifting overpass time is
handled upstream.

In [ ]:
# =============================================================================
# Cell 8 — Search ECOSTRESS and save the index (skipped when unavailable)
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    CFG.update(json.load(open(P("cfg_site.json"))))     # period, eco_possible, …
    roi = gpd.read_file(P("roi.geojson"))
    bbox = tuple(roi.total_bounds)

    ECO_PATTERNS = {"ET":  [r"ETdaily\.tif$", r"ETinst\.tif$", r"_ET\.tif$"],
                    "LST": [r"_LST\.tif$"]}
    ECO_QC = [r"_cloud\.tif$", r"_QC\.tif$"]

    if not CFG["eco_possible"]:
        eco_index = pd.DataFrame(columns=["gid", "local", "var", "qc", "hour"])
        eco_index.to_csv(P("eco_index.csv"), index=False)
        print(f"ECOSTRESS skipped: period ends {CFG['date_end']}, before the first "
              f"acquisitions ({CFG['eco_start']}). Optical-only cube.")
    elif globals().get("auth") is None and have("eco_index.csv"):
        eco_index = pd.read_csv(P("eco_index.csv"), parse_dates=["local"])
        print(f"Earthdata offline — using archived eco_index.csv ({len(eco_index)} granules)")
    else:
        t_start = max(pd.Timestamp(CFG["date_start"]), pd.Timestamp(CFG["eco_start"]))
        granules = earthaccess.search_data(
            short_name=CFG["eco_product"], bounding_box=bbox,
            temporal=(str(t_start.date()), CFG["date_end"]), count=-1)
        rows = []
        for g in granules:
            links = g.data_links()
            var_url = next((l for p in ECO_PATTERNS[CFG["eco_var"]]
                            for l in links if re.search(p, l)), None)
            if not var_url:
                continue
            qc_url = next((l for p in ECO_QC for l in links if re.search(p, l)), None)
            try:
                t = pd.to_datetime(g["umm"]["TemporalExtent"]["RangeDateTime"]
                                   ["BeginningDateTime"]).tz_localize(None)
            except Exception:
                continue
            rows.append(dict(gid=g["meta"]["native-id"],
                             local=t + pd.Timedelta(hours=CFG["utc_offset"]),
                             var=var_url, qc=qc_url))
        eco_index = pd.DataFrame(rows, columns=["gid", "local", "var", "qc"])
        if len(eco_index):
            eco_index = eco_index.sort_values("local").reset_index(drop=True)
            eco_index["hour"] = eco_index.local.dt.hour + eco_index.local.dt.minute / 60
        else:
            eco_index["hour"] = []
        eco_index.to_csv(P("eco_index.csv"), index=False)
        print(f"{CFG['eco_product']} / {CFG['eco_var']}: {len(eco_index)} granules "
              f"from {str(t_start.date())} → {CFG['date_end']}")
        if len(eco_index):
            print(f"  overpass hours : {eco_index.hour.min():.1f} – {eco_index.hour.max():.1f} local")
            print(f"  with QC layer  : {eco_index['qc'].notna().sum()}")
        print("  saved          : eco_index.csv")

## Cell 9 — Download ECOSTRESS (skipped when there is nothing to download)

Each granule is clipped to the ROI, QC-masked, reprojected onto the 70 m grid
unconditionally (shape-matching alone lets a differently-originated tile
through), cached as NetCDF and mirrored to the Data Store.

In [ ]:
# =============================================================================
# Cell 9 — Download ECOSTRESS onto the reference grid (resumable, mirrored)
# =============================================================================
def open_clip(handle, roi, masked=True):
    """Open a remote GeoTIFF and clip it to the ROI in its own CRS."""
    da = rxr.open_rasterio(handle, masked=masked).squeeze(drop=True)
    return da.rio.clip(roi.to_crs(da.rio.crs).geometry, from_disk=True)

def download_eco(index, cache_dir, grid, roi, var, max_retry=3):
    """Fetch one ECOSTRESS variable, QC-mask it, put it on `grid`, cache it."""
    t0, new, hit, failed = time.time(), 0, 0, 0
    for i, r in index.iterrows():
        path = os.path.join(cache_dir, f"{var}_{r.gid}.nc")
        if os.path.exists(path):
            hit += 1
            continue
        ok = False
        for attempt in range(max_retry):
            handles = None
            try:
                urls = [r["var"]] + ([r.qc] if isinstance(r.qc, str) else [])
                handles = earthaccess.open(urls)
                val = open_clip(handles[0], roi).rio.reproject_match(grid)
                if len(handles) > 1:
                    qc = open_clip(handles[1], roi, masked=False).rio.reproject_match(grid)
                    bad = np.nan_to_num(qc.values, nan=1) > 0
                    val = val.where(~xr.DataArray(bad, coords=val.coords, dims=val.dims))
                val.name = var
                val.attrs = dict(gid=r.gid, local_time=str(r.local))   # no inherited scale_factor
                val.encoding = {}
                val.to_netcdf(path)
                mirror(path)
                new, ok = new + 1, True
                break
            except Exception:
                time.sleep(5 * 2 ** attempt)
            finally:
                for h in handles or []:
                    try: h.close()
                    except Exception: pass
        if not ok:
            failed += 1
        if new and new % 25 == 0:
            print(f"  {i+1}/{len(index)}  downloaded {new}  ({(time.time()-t0)/60:.1f} min)")
        gc.collect()
    print(f"\n{var}: {new} new, {hit} cached, {failed} failed ({(time.time()-t0)/60:.1f} min)")


for SITE in SITES:
    use_site(SITE)
    eco_index = pd.read_csv(P("eco_index.csv"), parse_dates=["local"])
    roi = gpd.read_file(P("roi.geojson"))
    grid70 = load_da(P("grid_70m.nc")).rio.write_crs(roi.estimate_utm_crs())






    if not len(eco_index):
        print("no ECOSTRESS granules for this site/period — nothing to download")
    elif globals().get("auth") is None and not os.listdir(CACHE_ECO) and have("fused_cube.nc"):
        print("offline and fused_cube.nc restored — skipping ECOSTRESS download")
    else:
        download_eco(eco_index, CACHE_ECO, grid70, roi, CFG["eco_var"])
        fr = []
        for fn in sorted(os.listdir(CACHE_ECO))[:60]:
            if fn.endswith(".nc"):
                fr.append(float(np.isfinite(load_da(os.path.join(CACHE_ECO, fn)).values).mean()))
        if fr:
            fr = np.array(fr)
            print(f"cached granules sampled: {len(fr)} — near-full (>95%): {(fr >= 0.95).sum()}, "
                  f"best coverage {fr.max():.1%} (must approach 100%)")

## Cell 10 — Fuse into an evenly-spaced cube

NDVI and NDTI: **median** per window (a cloud Fmask missed is an outlier).
ET: **mean** per window, kept on its native 70 m grid, only when scenes exist.
Gaps are filled by linear interpolation along time; the share is printed and a
per-window `n_obs` record is kept so features can be weighted by evidence.

In [ ]:
# =============================================================================
# Cell 10 — Composite to windows, fill temporal gaps, save the cube
# =============================================================================
def assign_windows(times, windows):
    """Map observation times to window ids; -1 = outside every window."""
    t = pd.DatetimeIndex(times)
    out = np.full(len(t), -1, dtype=int)
    for _, w in windows.iterrows():
        out[(t >= w.win_start) & (t < w.win_next)] = w.win_id
    return out

def composite(values, win_idx, n_win, how):
    """Aggregate (n_obs, y, x) into (n_win, y, x). Empty windows stay NaN."""
    fn = {"median": np.nanmedian, "mean": np.nanmean}[how]
    out = np.full((n_win,) + values.shape[1:], np.nan, "float32")
    for w in range(n_win):
        sel = np.where(win_idx == w)[0]
        if len(sel):
            with np.errstate(all="ignore"):
                out[w] = fn(values[sel], axis=0)
    return out

def fill_time_gaps(cube, label):
    """Linear interpolation along time, per pixel; observed values untouched."""
    n = cube.shape[0]
    before = np.isfinite(cube).mean()
    filled = (pd.DataFrame(cube.reshape(n, -1))
                .interpolate(axis=0, limit_direction="both")
                .values.reshape(cube.shape).astype("float32"))
    observed = np.isfinite(cube)
    assert np.allclose(filled[observed], cube[observed])
    print(f"  {label:<5} valid {before:.1%} → {np.isfinite(filled).mean():.1%} "
          f"(interpolated {np.isfinite(filled).mean() - before:.1%})")
    return filled


for SITE in SITES:
    use_site(SITE)
    CFG.update(json.load(open(P("cfg_site.json"))))
    windows = pd.read_csv(P("windows.csv"), parse_dates=["win_start", "win_next",
                                                         "win_end", "win_center"])
    with xr.open_dataset(P("hls_scenes.nc")) as _s:
        scenes = _s.load()








    n_win = len(windows)
    print(f"compositing into {n_win} windows ({', '.join(f'{d} d' for d in sorted(windows.win_days.unique(), reverse=True))}):")
    idx = assign_windows(scenes.time.values, windows)
    cubes, observed = {}, {}
    for v in ("ndvi", "ndti"):
        cube = composite(scenes[v].values, idx, n_win, "median")
        observed[v] = np.isfinite(cube)                  # before interpolation
        print(f"  {v.upper():<5} {scenes.sizes['time']} obs days → "
              f"{int((~np.isnan(cube).all(axis=(1,2))).sum())}/{n_win} windows with data")
        cubes[v] = fill_time_gaps(cube, v.upper())

    # number of clear observation days per window, per pixel — the evidence behind
    # each composited value; 0 means the value is pure interpolation
    n_obs = np.zeros((n_win,) + scenes.ndvi.shape[1:], "int16")
    finite = np.isfinite(scenes.ndvi.values)
    for w in range(n_win):
        sel = np.where(idx == w)[0]
        if len(sel):
            n_obs[w] = finite[sel].sum(0)

    data_vars = {"ndvi": (("time", "y", "x"), cubes["ndvi"]),
                 "ndti": (("time", "y", "x"), cubes["ndti"]),
                 "n_obs": (("time", "y", "x"), n_obs)}
    grid30 = load_da(P("grid_30m.nc"))
    coords = {"time": pd.DatetimeIndex(windows.win_center),
              "y": grid30.y.values, "x": grid30.x.values}

    # --- ET, only when scenes exist --------------------------------------------
    eco_files = sorted(f for f in os.listdir(CACHE_ECO) if f.endswith(".nc"))
    if eco_files:
        eco_index = pd.read_csv(P("eco_index.csv"), parse_dates=["local"])
        gid_time = dict(zip(eco_index.gid, eco_index.local))
        vals, times = [], []
        for fn in eco_files:
            gid = fn[len(CFG["eco_var"]) + 1:-3]
            if gid not in gid_time:
                continue
            da = load_da(os.path.join(CACHE_ECO, fn))
            if np.isfinite(da.values).any():
                vals.append(da.values.astype("float32")); times.append(gid_time[gid])
        if vals:
            et_cube = composite(np.stack(vals), assign_windows(times, windows), n_win, "mean")
            print(f"  ET    {len(vals)} scenes → "
                  f"{int((~np.isnan(et_cube).all(axis=(1,2))).sum())}/{n_win} windows with data")
            et_cube = fill_time_gaps(et_cube, "ET")
            grid70 = load_da(P("grid_70m.nc"))
            data_vars["et"] = (("time", "y70", "x70"), et_cube)
            coords.update(y70=grid70.y.values, x70=grid70.x.values)
    else:
        print("  ET    no scenes — optical-only cube")

    fused = xr.Dataset(data_vars, coords=coords,
                       attrs={"site": SITE, "x_days": str(sorted(windows.win_days.unique())),
                              "crs": str(gpd.read_file(P("roi.geojson")).estimate_utm_crs()),
                              "created": pd.Timestamp.now().isoformat(timespec="seconds")})
    if os.path.exists(P("fused_cube.nc")):
        os.remove(P("fused_cube.nc"))
    fused.to_netcdf(P("fused_cube.nc"))
    print(f"\nfused cube: {n_win} time steps — " + ", ".join(
          f"{v} {tuple(fused[v].shape)}" for v in fused.data_vars))
    print(f"saved: fused_cube.nc  ({os.path.getsize(P('fused_cube.nc'))/1e6:.1f} MB)")

## Cell 11 — Ground-truth fields on the grid

The site's fields (from the store, not from CSB) are burned onto the 30 m grid
with the pixel-centre rule. Two rasters: `field_id` (row order in
`fields_site.gpkg`, 0 = outside) and `cropland` (any field). Field pixels are
also shrunk by one pixel from the boundary for the per-field statistics —
CSB-derived boundaries are ±1 pixel and the mixed rim would otherwise blur the
contrast the labels are about.

In [ ]:
# =============================================================================
# Cell 11 — Burn the ground-truth fields onto the 30 m grid
# =============================================================================
from scipy import ndimage


for SITE in SITES:
    use_site(SITE)
    fields = gpd.read_file(P("fields_site.gpkg"))
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(
        gpd.read_file(P("roi.geojson")).estimate_utm_crs())
    fields_utm = fields.to_crs(grid30.rio.crs)
    fields_utm["field_id"] = np.arange(1, len(fields_utm) + 1, dtype="int32")

    shape = (grid30.sizes["y"], grid30.sizes["x"])
    fid = features.rasterize(zip(fields_utm.geometry, fields_utm.field_id),
                             out_shape=shape, transform=grid30.rio.transform(),
                             fill=0, dtype="int32", all_touched=False)

    # inner core: drop the one-pixel rim of every field
    core = np.zeros_like(fid)
    for i in fields_utm.field_id:
        m = fid == i
        core[ndimage.binary_erosion(m, structure=np.ones((3, 3)))] = i

    for name, arr in (("field_id", fid), ("field_core", core),
                      ("cropland", (fid > 0).astype("int32"))):
        (xr.DataArray(arr.astype("int32"), coords=grid30.coords, dims=grid30.dims)
           .rio.write_crs(grid30.rio.crs).rio.write_nodata(None)
           .rio.to_raster(P(f"{name}.tif"), compress="DEFLATE"))
    fields_utm[["field_uid", "role", "field_id", "geometry"]].to_file(
        P("fields_site_utm.gpkg"), driver="GPKG")

    px = np.bincount(fid.ravel(), minlength=len(fields_utm) + 1)[1:]
    pxc = np.bincount(core.ravel(), minlength=len(fields_utm) + 1)[1:]
    print(f"{len(fields_utm)} fields burned onto {shape[0]} x {shape[1]} px")
    for (_, f), n, nc in zip(fields_utm.iterrows(), px, pxc):
        print(f"  {f.field_uid:<26} {f.role:<20} {n:>5} px  core {nc:>5} px  "
              f"({n * 0.09:.1f} ha)")
    assert (pxc > 0).all(), "a field has no core pixels — too small for 30 m analysis"
    print("saved: field_id.tif, field_core.tif, cropland.tif, fields_site_utm.gpkg")

## Cell 12 — Look at it: maps and field trajectories

Top row: NDVI in the strongest labelled window of the latest cover-crop
season, NDVI at the summer peak, NDTI in the same spring window, with field
outlines coloured by role. Bottom: field-mean NDVI over the whole period, one
line per field, cover-crop fields in green and controls in grey; every
labelled detection window is shaded and every known termination date marked.

This is where a label that contradicts the imagery first shows up — read it
before trusting any number from Cell 14.

In [ ]:
# =============================================================================
# Cell 12 — QC maps and per-field trajectories
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    CFG.update(json.load(open(P("cfg_site.json"))))
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    labels = pd.read_csv(P("seasons_site.csv"),
                         parse_dates=["detect_start", "detect_end", "best_start",
                                      "best_end", "cc_termination_date"])
    fid  = rxr.open_rasterio(P("field_id.tif")).squeeze(drop=True).values
    core = rxr.open_rasterio(P("field_core.tif")).squeeze(drop=True).values
    t = pd.DatetimeIndex(fused.time.values)
    ndvi, ndti = fused.ndvi.values, fused.ndti.values

    ROLE_COLOR = {"cc_treatment": "#2b7a3d", "candidate_same_farm": "#7fbf7b",
                  "nt_control": "#8a99a5", "neighbour_control": "#8a99a5",
                  "trial_host_field": "#c4763a"}
    edge = np.zeros_like(fid, bool)
    edge[:, 1:] |= fid[:, 1:] != fid[:, :-1]
    edge[1:, :] |= fid[1:, :] != fid[:-1, :]
    edge_img = np.ma.masked_where(~edge, np.ones_like(fid, "float32"))

    # --- pick display windows from the labels ----------------------------------
    pos = labels[labels.cc_present == 1]
    ref = pos if len(pos) else labels
    last = ref.sort_values("detect_end").iloc[-1]
    spring_t = last.best_end if pd.notna(last.best_end) else last.detect_end
    i_spring = int(np.argmin(np.abs(t - spring_t)))
    summer_t = last.detect_end + pd.Timedelta(days=75)
    i_summer = int(np.argmin(np.abs(t - summer_t)))

    # --- per-field mean trajectories (core pixels) -----------------------------
    traj = {}
    for _, f in fields.iterrows():
        m = core == f.field_id
        traj[f.field_uid] = np.nanmean(ndvi[:, m], axis=1)

    fig = plt.figure(figsize=(16, 11))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 0.9])
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    for a, img, ttl, cm, vr in [
            (axes[0], ndvi[i_spring], f"NDVI — labelled spring window\n{t[i_spring]:%Y-%m-%d}",
             "RdYlGn", (0, 0.8)),
            (axes[1], ndvi[i_summer], f"NDVI — summer\n{t[i_summer]:%Y-%m-%d}", "RdYlGn", (0, 0.9)),
            (axes[2], ndti[i_spring], f"NDTI — same spring window\n{t[i_spring]:%Y-%m-%d}",
             "copper_r", (0, 0.25))]:
        im = a.imshow(img, cmap=cm, vmin=vr[0], vmax=vr[1])
        a.imshow(edge_img, cmap="gray_r", vmin=0, vmax=1, alpha=0.6, interpolation="nearest")
        for _, f in fields.iterrows():                      # label each field
            yy, xx = np.where(fid == f.field_id)
            if len(yy):
                a.text(xx.mean(), yy.mean(), f.field_uid.split("_")[-1], fontsize=7,
                       ha="center", va="center", color=ROLE_COLOR.get(f.role, "k"),
                       bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.5))
        a.set_title(ttl, fontsize=10); a.set_xticks([]); a.set_yticks([])
        plt.colorbar(im, ax=a, fraction=0.046)

    ax = fig.add_subplot(gs[1, :])
    for _, f in fields.iterrows():
        c = ROLE_COLOR.get(f.role, "k")
        ax.plot(t, traj[f.field_uid], "-", lw=1.3 if "cc" in f.role or "candidate" in f.role else 0.9,
                color=c, alpha=0.9 if f.role == "cc_treatment" else 0.6,
                label=f"{f.field_uid.split('_')[-1]} ({f.role})")
    for _, r in labels.iterrows():
        ax.axvspan(r.detect_start, r.detect_end,
                   color="#2b7a3d" if r.cc_present == 1 else "#8a99a5", alpha=0.06)
        if pd.notna(r.cc_termination_date):
            ax.axvline(r.cc_termination_date, color="#c44", lw=0.8, ls="--")
    ax.set_ylabel("field-mean NDVI (core pixels)")
    ax.set_title(f"{SITE}: field trajectories — shaded = labelled detection windows "
                 f"(green CC / grey no CC), red dashes = known termination dates", fontsize=10)
    ax.grid(alpha=0.25)
    if len(fields) <= 12:
        ax.legend(fontsize=7, ncol=3, loc="upper left")
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"qc_{SITE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"saved: {fig_path}")

    # quick numbers behind the picture
    for _, r in pos.iterrows():
        sel = (t >= (r.best_start if pd.notna(r.best_start) else r.detect_end - pd.Timedelta(days=42))) \
              & (t <= r.detect_end)
        ctrl = [u for u, f in zip(fields.field_uid, fields.role) if "control" in f]
        v_cc = np.nanmean(traj[r.field_uid][sel]) if sel.any() else np.nan
        v_ct = np.nanmean([np.nanmean(traj[u][sel]) for u in ctrl]) if ctrl and sel.any() else np.nan
        print(f"  {r.field_uid:<26} {r.season}  spring NDVI {v_cc:.2f}  vs controls {v_ct:.2f}")

## Cell 13 — Features per field × season

The five phenology features of v5, computed on each field's mean NDVI series
and anchored on that season's own labels rather than fixed calendar dates:

| feature | window |
|---|---|
| `n_green`, `ndvi_int` | `detect_start` → `detect_end` (harvest → before termination); `n_green` = days with NDVI > 0.30, so 8- and 16-day windows compare |
| `up_slope` | the 75 days before termination |
| `term_drop` | 24 days before → 40 days after termination |
| `peak_win` | days from `detect_start` to the NDVI peak before termination |

`term` is `cc_termination_date` when the paper gives one, else `detect_end`.
Three more that the labelled sites make possible: `best_ndvi` (mean NDVI in the
strongest window), `best_ndvi_rel` (the same minus the site median of all
fields in that window — a label-free reference that cancels the season's
weather), and `n_obs_off` (clear observation days behind the off-season
values, so a feature built on interpolation can be recognised as such).

In [ ]:
# =============================================================================
# Cell 13 — Per field × season features, joined to the labels
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    CFG.update(json.load(open(P("cfg_site.json"))))
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    labels = pd.read_csv(P("seasons_site.csv"),
                         parse_dates=["detect_start", "detect_end", "best_start",
                                      "best_end", "cc_termination_date", "next_cash_plant"])
    core = rxr.open_rasterio(P("field_core.tif")).squeeze(drop=True).values
    t = pd.DatetimeIndex(fused.time.values)
    win_days = pd.read_csv(P("windows.csv")).win_days.values      # 16 or 8 per window
    assert len(win_days) == len(t)
    GREEN = 0.30                                   # provisional; Cell 14 sweeps it

    # field-mean series on core pixels
    series, nobs = {}, {}
    for _, f in fields.iterrows():
        m = core == f.field_id
        series[f.field_uid] = {v: np.nanmean(fused[v].values[:, m], axis=1) for v in ("ndvi", "ndti")}
        nobs[f.field_uid] = fused.n_obs.values[:, m].max(axis=1)     # best pixel per window
        if "et" in fused:
            # ET on its own 70 m grid: sample at the field's 30 m core centroid
            yy, xx = np.where(m)
            cy = float(fused.y.values[int(yy.mean())]); cx = float(fused.x.values[int(xx.mean())])
            iy = int(np.argmin(np.abs(fused.y70.values - cy)))
            ix = int(np.argmin(np.abs(fused.x70.values - cx)))
            series[f.field_uid]["et"] = fused.et.values[:, iy, ix]
    site_median = np.nanmedian(np.stack([s["ndvi"] for s in series.values()]), axis=0)

    rows = []
    for _, r in labels.iterrows():
        s = series.get(r.field_uid)
        if s is None:
            continue
        nd = s["ndvi"]
        term = r.cc_termination_date if pd.notna(r.cc_termination_date) else r.detect_end
        off    = (t >= r.detect_start) & (t <= r.detect_end)
        rise   = (t >= term - pd.Timedelta(days=75)) & (t <= term)
        tw     = (t >= term - pd.Timedelta(days=24)) & (t <= term + pd.Timedelta(days=40))
        best   = ((t >= (r.best_start if pd.notna(r.best_start) else r.detect_end - pd.Timedelta(days=42)))
                  & (t <= (r.best_end if pd.notna(r.best_end) else r.detect_end)))
        pre    = (t >= r.detect_start) & (t <= term)
        d = np.diff(nd)
        feat = dict(
            field_uid=r.field_uid, season=r.season, cc_present=int(r.cc_present),
            role=fields.set_index("field_uid").role.get(r.field_uid),
            label_source=r.label_source, label_confidence=r.label_confidence,
            n_win_off=int(off.sum()),
            n_obs_off=int(nobs[r.field_uid][off].sum()) if off.any() else 0,
            n_green=int(win_days[off][nd[off] > GREEN].sum()) if off.any() else np.nan,   # days
            ndvi_int=float(np.nansum(nd[off] * win_days[off]) / 100) if off.any() else np.nan,
            off_ndvi_mean=float(np.nanmean(nd[off])) if off.any() else np.nan,
            best_ndvi=float(np.nanmean(nd[best])) if best.any() else np.nan,
            best_ndvi_rel=float(np.nanmean(nd[best] - site_median[best])) if best.any() else np.nan,
            best_ndti=float(np.nanmean(s["ndti"][best])) if best.any() else np.nan,
            up_slope=float(np.nanmax(d[rise[1:]])) if rise[1:].any() else np.nan,
            term_drop=float(-np.nanmin(d[tw[1:]])) if tw[1:].any() else np.nan,
            peak_win=float((t[pre][int(np.nanargmax(nd[pre]))] - r.detect_start).days) if pre.any() else np.nan,
            et_best=float(np.nanmean(s["et"][best])) if ("et" in s and best.any()) else np.nan,
            term_known=bool(pd.notna(r.cc_termination_date)),
        )
        rows.append(feat)

    feat_tbl = pd.DataFrame(rows)
    feat_tbl.to_csv(P("features_by_season.csv"), index=False)
    print(f"{len(feat_tbl)} field-seasons → features_by_season.csv")
    show = ["field_uid", "season", "cc_present", "n_obs_off", "n_green", "ndvi_int",
            "best_ndvi", "best_ndvi_rel", "up_slope", "term_drop", "peak_win", "et_best"]
    with pd.option_context("display.width", 200, "display.float_format", "{:.3f}".format):
        print(feat_tbl[show].sort_values(["cc_present", "field_uid", "season"]).to_string(index=False))

## Cell 14 — Do the features separate the labels?

For every feature: AUC (rank-based, label 1 = cover crop), the class medians,
and the best single threshold by Youden's J. With a dozen or two dozen
field-seasons per site the AUCs carry wide uncertainty — the point is the
ordering of features and whether any of them is clearly informative, not the
third decimal. Labels inferred from NDVI (`label_source = ndvi_inferred`) are
circular for NDVI features and are flagged in the table.

In [ ]:
# =============================================================================
# Cell 14 — Separability of each feature on this site's labels
# =============================================================================
def auc(pos, neg):
    """Mann–Whitney AUC: P(score_pos > score_neg), ties count half."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if not len(pos) or not len(neg):
        return np.nan
    gt = (pos[:, None] > neg[None, :]).sum(); eq = (pos[:, None] == neg[None, :]).sum()
    return (gt + 0.5 * eq) / (len(pos) * len(neg))

def youden(pos, neg):
    """Threshold maximising sensitivity + specificity − 1 (score > thr → CC)."""
    vals = np.sort(np.unique(np.concatenate([pos, neg])))
    vals = vals[np.isfinite(vals)]
    best = (-1, np.nan, 0, 0)
    for thr in vals:
        se = (pos > thr).mean(); sp = (neg <= thr).mean()
        if se + sp - 1 > best[0]:
            best = (se + sp - 1, thr, se, sp)
    return best


for SITE in SITES:
    use_site(SITE)
    feat_tbl = pd.read_csv(P("features_by_season.csv"))
    FEATS = ["best_ndvi_rel", "best_ndvi", "n_green", "ndvi_int", "off_ndvi_mean",
             "up_slope", "term_drop", "peak_win", "best_ndti", "et_best"]






    pos_all = feat_tbl[feat_tbl.cc_present == 1]; neg_all = feat_tbl[feat_tbl.cc_present == 0]
    print(f"{SITE}: {len(pos_all)} CC vs {len(neg_all)} no-CC field-seasons  "
          f"(label sources: {feat_tbl.label_source.value_counts().to_dict()})\n")
    print(f"{'feature':<15}{'AUC':>6}{'median CC':>11}{'median no':>11}{'thr (J)':>10}{'sens':>6}{'spec':>6}")
    res = []
    for f in FEATS:
        p, n = pos_all[f].values, neg_all[f].values
        if np.isfinite(p).sum() == 0 or np.isfinite(n).sum() == 0:
            continue
        a = auc(p, n); j, thr, se, sp = youden(p[np.isfinite(p)], n[np.isfinite(n)])
        res.append(dict(feature=f, auc=a, med_cc=np.nanmedian(p), med_no=np.nanmedian(n),
                        thr=thr, sens=se, spec=sp))
        print(f"{f:<15}{a:>6.2f}{np.nanmedian(p):>11.3f}{np.nanmedian(n):>11.3f}"
              f"{thr:>10.3f}{se:>6.2f}{sp:>6.2f}")
    pd.DataFrame(res).to_csv(P("feature_auc.csv"), index=False)

    # strip plots: every field-season as a point, class by colour
    fig, ax = plt.subplots(2, 5, figsize=(16, 6.5)); ax = ax.ravel()
    rng = np.random.default_rng(0)
    for a, f in zip(ax, FEATS):
        for cls, col, x0 in ((0, "#8a99a5", 0), (1, "#2b7a3d", 1)):
            v = feat_tbl.loc[feat_tbl.cc_present == cls, f].values
            a.scatter(x0 + rng.normal(0, 0.06, len(v)), v, s=22, color=col, alpha=0.8)
        circ = " (circular)" if (feat_tbl.label_source.astype(str).str.contains("ndvi").any()
                                 and "ndvi" in f) else ""
        a.set_title(f"{f}{circ}", fontsize=9); a.set_xticks([0, 1]); a.set_xticklabels(["no CC", "CC"])
        a.grid(alpha=0.25)
    plt.suptitle(f"{SITE}: feature values by label ({len(pos_all)} CC / {len(neg_all)} no CC)")
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"features_{SITE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"\nsaved: feature_auc.csv, {fig_path}")
    if feat_tbl.label_source.astype(str).str.contains("ndvi").any():
        print("note: some labels here were inferred from NDVI — AUCs of NDVI features on "
              "those rows are circular and overstate real skill")

## Cell 15 — Archive to the Data Store

Products, figures, the feature tables and both per-scene caches go to
`ground_truth_runs/<site>/`. Files already present with the same size are not
copied again, so this is quick after the mirrored downloads.

In [ ]:
# =============================================================================
# Cell 15 — Copy this site's products to the Data Store
# =============================================================================
def put(src, rel):
    dst = os.path.join(ARCHIVE, rel)
    mb = os.path.getsize(src) / 1e6
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        return False, mb
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    return True, mb


for SITE in SITES:
    use_site(SITE)
    os.makedirs(ARCHIVE, exist_ok=True)
    PRODUCTS = ["roi.geojson", "windows.csv", "cfg_site.json", "fields_site.gpkg",
                "fields_site_utm.gpkg", "seasons_site.csv", "grid_30m.nc", "grid_70m.nc",
                "hls_index.csv", "eco_index.csv", "hls_scenes.nc", "fused_cube.nc",
                "field_id.tif", "field_core.tif", "cropland.tif",
                "features_by_season.csv", "feature_auc.csv"]




    t0, copied, skipped, missing = time.time(), [], 0, []
    for name in PRODUCTS:
        if not os.path.exists(P(name)):
            missing.append(name); continue
        did, mb = put(P(name), name)
        copied.append((name, mb)) if did else None; skipped += (not did)
    for sub in ("figures", "cache/hls_v2", "cache/eco"):
        d = P(sub); n = n_new = 0
        if os.path.isdir(d):
            for f in sorted(os.listdir(d)):
                did, mb = put(os.path.join(d, f), os.path.join(sub, f)); n += 1; n_new += did
        if n:
            copied.append((f"{sub}/  ({n_new} new of {n})", 0.0))
    print(f"archived to {ARCHIVE}   ({(time.time()-t0)/60:.1f} min)")
    for name, mb in copied:
        print(f"  {name:<40} {mb:>8.2f} MB" if mb else f"  {name}")
    print(f"  {'already archived, unchanged':<40} {skipped:>5} file(s)")
    if missing:
        print(f"not found (run the earlier cells): {missing}")

## Cell 16 — All sites on one page

One row per site: NDVI in that site's strongest labelled spring window with
the field outlines, the field-mean NDVI trajectories over the site's own period
(labelled detection windows shaded, known termination dates marked), and the
AUC of every feature on that site's labels. The row header carries the site,
its location, the period, the window length and the label counts, so the
figure stands on its own. The per-site feature and AUC tables are also
concatenated into one file each.

In [ ]:
# =============================================================================
# Cell 16 — Combined figure and tables across all sites
# =============================================================================
SUMMARY = os.path.join(CFG["work_root"], "_summary")
os.makedirs(SUMMARY, exist_ok=True)
ROLE_COLOR = {"cc_treatment": "#2b7a3d", "candidate_same_farm": "#7fbf7b",
              "nt_control": "#8a99a5", "neighbour_control": "#8a99a5",
              "trial_host_field": "#c4763a"}

done = [s for s in SITES if os.path.isfile(os.path.join(CFG["work_root"], s, "feature_auc.csv"))]
assert done, "no site has reached Cell 14 yet"
feat_all, auc_all = [], []
fig, axes = plt.subplots(len(done), 3, figsize=(19, 4.8 * len(done)),
                         gridspec_kw={"width_ratios": [1, 2.2, 1.1]}, squeeze=False)

for k, site in enumerate(done):
    use_site(site, quiet=True)
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    labels = pd.read_csv(P("seasons_site.csv"),
                         parse_dates=["detect_start", "detect_end", "best_start",
                                      "best_end", "cc_termination_date"])
    windows = pd.read_csv(P("windows.csv"))
    fid  = rxr.open_rasterio(P("field_id.tif")).squeeze(drop=True).values
    core = rxr.open_rasterio(P("field_core.tif")).squeeze(drop=True).values
    auc_tbl = pd.read_csv(P("feature_auc.csv")); auc_tbl.insert(0, "site", site)
    ft = pd.read_csv(P("features_by_season.csv")); ft.insert(0, "site", site)
    feat_all.append(ft); auc_all.append(auc_tbl)
    t = pd.DatetimeIndex(fused.time.values)
    ndvi = fused.ndvi.values
    cen = gpd.read_file(P("roi.geojson")).geometry.centroid.iloc[0]

    # --- header text for this row ------------------------------------------
    n_cc, n_no = int((labels.cc_present == 1).sum()), int((labels.cc_present == 0).sum())
    wd = " + ".join(f"{d} d" for d in sorted(windows.win_days.unique(), reverse=True) if d in (8, 16))
    hdr = (f"{site}   {abs(cen.y):.3f}°{'N' if cen.y >= 0 else 'S'} {abs(cen.x):.3f}°{'W' if cen.x < 0 else 'E'}   "
           f"{len(fields)} fields · {CFG['date_start']} → {CFG['date_end']} · windows {wd}"
           f" · {n_cc} CC / {n_no} no-CC field-seasons · ET {'yes' if 'et' in fused else 'no'}")

    # --- panel 1: NDVI map, strongest labelled spring window -----------------
    pos = labels[labels.cc_present == 1]
    last = (pos if len(pos) else labels).sort_values("detect_end").iloc[-1]
    spring_t = last.best_end if pd.notna(last.best_end) else last.detect_end
    i_sp = int(np.argmin(np.abs(t - spring_t)))
    edge = np.zeros_like(fid, bool)
    edge[:, 1:] |= fid[:, 1:] != fid[:, :-1]; edge[1:, :] |= fid[1:, :] != fid[:-1, :]
    a = axes[k, 0]
    im = a.imshow(ndvi[i_sp], cmap="RdYlGn", vmin=0, vmax=0.8)
    a.imshow(np.ma.masked_where(~edge, np.ones_like(fid, "float32")), cmap="gray_r",
             vmin=0, vmax=1, alpha=0.6, interpolation="nearest")
    for _, f in fields.iterrows():
        yy, xx = np.where(fid == f.field_id)
        if len(yy):
            a.text(xx.mean(), yy.mean(), f.field_uid.split("_")[-1], fontsize=7, ha="center",
                   va="center", color=ROLE_COLOR.get(f.role, "k"),
                   bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.5))
    a.set_title(f"NDVI {t[i_sp]:%Y-%m-%d} (spring, season {last.season})", fontsize=9)
    a.set_xticks([]); a.set_yticks([])
    plt.colorbar(im, ax=a, fraction=0.046)

    # --- panel 2: field trajectories ----------------------------------------
    a = axes[k, 1]
    for _, f in fields.iterrows():
        m = core == f.field_id
        a.plot(t, np.nanmean(ndvi[:, m], axis=1), "-", color=ROLE_COLOR.get(f.role, "k"),
               lw=1.3 if f.role in ("cc_treatment", "candidate_same_farm") else 0.9,
               alpha=0.9 if f.role == "cc_treatment" else 0.6,
               label=f"{f.field_uid.split('_')[-1]} ({f.role})")
    for _, r in labels.iterrows():
        a.axvspan(r.detect_start, r.detect_end,
                  color="#2b7a3d" if r.cc_present == 1 else "#8a99a5", alpha=0.05)
        if pd.notna(r.cc_termination_date):
            a.axvline(r.cc_termination_date, color="#c44", lw=0.8, ls="--")
    a.set_ylabel("field-mean NDVI"); a.grid(alpha=0.25)
    a.set_title(hdr, fontsize=9.5, loc="left")
    if len(fields) <= 12:
        a.legend(fontsize=6.5, ncol=3, loc="upper left")

    # --- panel 3: AUC per feature -------------------------------------------
    a = axes[k, 2]
    s = auc_tbl.sort_values("auc")
    cols = ["#2b7a3d" if v >= 0.5 else "#c4763a" for v in s.auc]
    a.barh(s.feature, s.auc, color=cols)
    a.axvline(0.5, color="k", lw=0.8); a.set_xlim(0, 1)
    circ = ft.label_source.astype(str).str.contains("ndvi").any()
    a.set_title("AUC per feature" + (" (NDVI labels → circular)" if circ else ""), fontsize=9)
    a.tick_params(axis="y", labelsize=8); a.grid(alpha=0.25, axis="x")

plt.tight_layout()
fig_path = os.path.join(SUMMARY, "all_sites_summary.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()

pd.concat(feat_all).to_csv(os.path.join(SUMMARY, "features_all_sites.csv"), index=False)
auc_wide = pd.concat(auc_all).pivot(index="feature", columns="site", values="auc")
auc_wide.to_csv(os.path.join(SUMMARY, "feature_auc_all_sites.csv"))
print(f"{len(done)} site(s) summarised → {fig_path}")
with pd.option_context("display.float_format", "{:.2f}".format):
    print("\nAUC by site:\n" + auc_wide.reindex(auc_all[0].feature).to_string())

if CFG["use_archive"]:
    dst = os.path.join(CFG["archive_root"], "_summary")
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(SUMMARY):
        shutil.copy2(os.path.join(SUMMARY, f), os.path.join(dst, f))
    print(f"copied to {dst}")

---
## Reading the results

**Positive labels at the IL and NE sites were partly inferred from NDVI**
(`label_source`). Any NDVI feature evaluated on those rows is circular; the
Indiana site (labels from the paper, boundaries hand-verified) is the clean
test.

**The evaluation is per site.** Pooling sites mixes different cover-crop
species, establishment methods and climates; do that only after each site
behaves on its own.

**Winter is when thermal data is weakest** and ET is sampled at one 70 m pixel
per field here — treat `et_best` as exploratory until fields larger than a few
ET pixels are evaluated.
